# continuity

> Analyze the continuity of the adjacent chips/quads.

In [ ]:
# | default_exp euclid.continuity

In [ ]:
# | exporti

from astropy.io import fits
import numpy as np
import warnings

from nicl.euclid.constants import NISP, VIS
from nicl.euclid.utilities import (
    get_dither_id_from_filename,
    get_obs_id_from_filename,
)
from nicl.utilities import bootstrap_median_error, fast_median_error

In [ ]:
# | hide
# additional imports for examples

from nicl.euclid.utilities import default_data_path

In [ ]:
# | export


def measure_edge_diff(fn, window=1):
    obsid = get_obs_id_from_filename(fn)
    dither_id = get_dither_id_from_filename(fn)
    instrument = VIS
    with fits.open(fn) as hdul:
        exts1 = []
        exts2 = []
        edge_diff = []
        edge_diff_err = []
        for ext in instrument.extnames:
            # figure out which adjecent chip/quad to use
            if instrument.name == "VIS":
                x = int(ext[0])
                y = int(ext[2])
                if y <= 3:
                    match ext[-1]:
                        case "E":
                            adj_top_ext = ext.replace("E", "H")
                            adj_right_ext = ext.replace("E", "F")
                        case "F":
                            adj_top_ext = ext.replace("F", "G")
                            adj_right_ext = None
                        case "G":
                            adj_top_ext = None
                            adj_right_ext = None
                        case "H":
                            adj_top_ext = None
                            adj_right_ext = ext.replace("H", "G")
                else:
                    match ext[-1]:
                        case "E":
                            adj_top_ext = None
                            adj_right_ext = None
                        case "F":
                            adj_top_ext = None
                            adj_right_ext = ext.replace("F", "E")
                        case "G":
                            adj_top_ext = ext.replace("G", "F")
                            adj_right_ext = ext.replace("G", "H")
                        case "H":
                            adj_top_ext = ext.replace("H", "E")
                            adj_right_ext = None
            elif instrument.name == "NISP":
                pass
            if adj_top_ext is not None or adj_right_ext is not None:
                sci_img = hdul[ext + ".SCI"].data
                # rms_img = hdul[ext + ".RMS"].data
                if instrument.name == "VIS":
                    dq_img = hdul[ext + ".FLG"].data
                elif instrument.name == "NISP":
                    dq_img = hdul[ext + ".DQ"].data
                bad_pix_mask = np.any(
                    [(dq_img & 2**bit > 0) for bit in instrument.bad_pix_bits],
                    axis=0,
                )
                # mask out bad pixels
                sci_img[bad_pix_mask] = np.nan
            else:
                continue

            # compute the edge difference
            if adj_right_ext is not None:
                right_edge = sci_img[:, -window:]
                # right_edge_rms = rms_img[:, -window:]
                left_edge = hdul[adj_right_ext + ".SCI"].data[:, :window]
                # left_edge_rms = hdul[adj_right_ext + ".RMS"].data[:, :window]
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message=".*All-NaN slice encountered")
                    left_edge_collapsed = np.nanmedian(left_edge, axis=1)
                    right_edge_collapsed = np.nanmedian(right_edge, axis=1)
                    # left_edge_rms_collapsed = bootstrap_median_error(
                    #     left_edge, left_edge_rms, axis=1
                    # )
                    # right_edge_rms_collapsed = bootstrap_median_error(
                    #     right_edge, right_edge_rms, axis=1
                    # )
                    # left_right_diff_rms = np.sqrt(
                    #     left_edge_rms_collapsed**2 + right_edge_rms_collapsed**2
                    # )
                    left_right_edge_diff = np.nanmedian(
                        left_edge_collapsed - right_edge_collapsed
                    )
                    # left_right_edge_diff_err = bootstrap_median_error(
                    #     left_edge_collapsed - right_edge_collapsed, left_right_diff_rms
                    # )
                    left_right_edge_diff_err = fast_median_error(
                        left_edge_collapsed - right_edge_collapsed
                    )
                exts1.append(adj_right_ext)
                exts2.append(ext)
                edge_diff.append(left_right_edge_diff)
                edge_diff_err.append(left_right_edge_diff_err)
            if adj_top_ext is not None:
                top_edge = sci_img[-window:, :]
                # top_edge_rms = rms_img[-window:, :]
                bottom_edge = hdul[adj_top_ext + ".SCI"].data[:window, :]
                # bottom_edge_rms = hdul[adj_top_ext + ".RMS"].data[:window, :]
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message=".*All-NaN slice encountered")
                    bottom_edge_collapsed = np.nanmedian(bottom_edge, axis=0)
                    top_edge_collapsed = np.nanmedian(top_edge, axis=0)
                    # bottom_edge_rms_collapsed = bootstrap_median_error(
                    #     bottom_edge, bottom_edge_rms, axis=0
                    # )
                    # top_edge_rms_collapsed = bootstrap_median_error(
                    #     top_edge, top_edge_rms, axis=0
                    # )
                    # bottom_top_diff_rms = np.sqrt(
                    #     bottom_edge_rms_collapsed**2 + top_edge_rms_collapsed**2
                    # )
                    bottom_top_edge_diff = np.nanmedian(
                        bottom_edge_collapsed - top_edge_collapsed
                    )
                    # bottom_top_edge_diff_err = bootstrap_median_error(
                    #     bottom_edge_collapsed - top_edge_collapsed, bottom_top_diff_rms
                    # )
                    bottom_top_edge_diff_err = fast_median_error(
                        bottom_edge_collapsed - top_edge_collapsed
                    )
                exts1.append(adj_top_ext)
                exts2.append(ext)
                edge_diff.append(bottom_top_edge_diff)
                edge_diff_err.append(bottom_top_edge_diff_err)
        return obsid, dither_id, exts1, exts2, edge_diff, edge_diff_err

## Examples

In [ ]:
obs_id = 2683
dither_id = "3-1"
fn = list(
    default_data_path("Q1_R1", "VIS_QUAD", str(obs_id)).glob(f"*{dither_id}*.fits")
)[0]
obs_id_, dither_id_, quad_id1, quad_id2, edge_diff, edge_diff_rms = measure_edge_diff(
    fn, window=10
)

In [ ]:
for quad1, quad2, diff, diff_rms in zip(quad_id1, quad_id2, edge_diff, edge_diff_rms):
    # if quad1 is not None and quad2 is not None:
    print(f"{quad1} - {quad2}: {diff} {diff_rms}")

1-1.F - 1-1.E: 0.5766620635986328 0.06906657597448314
1-1.H - 1-1.E: 0.24362754821777344 0.07289279256663685
1-1.G - 1-1.F: 0.059988975524902344 0.06300422837488702
1-1.G - 1-1.H: 0.20467758178710938 0.06030723087446495
1-2.F - 1-2.E: 0.5208950042724609 0.06606111498467154
1-2.H - 1-2.E: 0.1773233413696289 0.06864105917274692
1-2.G - 1-2.F: -0.4510326385498047 0.06726181509870671
1-2.G - 1-2.H: -0.21147537231445312 0.061593399238404754
1-3.F - 1-3.E: -0.11479759216308594 0.06383198204199955
1-3.H - 1-3.E: -0.02207183837890625 0.07893356250083106
1-3.G - 1-3.F: 0.48537540435791016 0.06578848128097349
1-3.G - 1-3.H: 0.6759462356567383 0.06450329139840003
1-4.E - 1-4.F: 0.07422447204589844 0.06141830049103791
1-4.H - 1-4.G: 0.27189159393310547 0.06207649568101593
1-4.F - 1-4.G: -0.2959146499633789 0.06845868754332433
1-4.E - 1-4.H: -0.6878395080566406 0.07066104011727133
1-5.E - 1-5.F: 0.12644195556640625 0.0625552109100001
1-5.H - 1-5.G: 0.6733331680297852 0.06167973020915248
1-5.F - 1-5

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()